# Analiza sieci społecznościowych społeczności muzycznych Reddita

Notebook przedstawia analizę relacji między użytkownikami wybranych muzycznych subredditów. Celem jest zbadanie, jak użytkownicy wchodzą ze sobą w interakcje, które osoby zajmują najbardziej centralne pozycje w strukturze dyskusji oraz czy w obrębie społeczności występują wyraźne klastry komunikacyjne.

## Założenia analizy sieciowej

W analizie sieciowej każdy użytkownik Reddita jest traktowany jako węzeł (*node*), a relacja odpowiedzi między użytkownikami jako krawędź (*edge*). Krawędź jest skierowana: `author -> parent_author`, co oznacza, że użytkownik `author` odpowiedział na komentarz użytkownika `parent_author`.

Wagi krawędzi (`weight`) oznaczają liczbę interakcji między tą samą parą użytkowników. Jeśli użytkownik A kilkukrotnie odpowiadał użytkownikowi B, relacja A → B otrzymuje większą wagę. Taka konstrukcja grafu pozwala badać zarówno ogólną aktywność użytkowników, jak i ich pozycję w strukturze rozmów.

Interpretacja grafu:

- większa liczba krawędzi oznacza większą intensywność interakcji,
- wysoki stopień węzła może świadczyć o częstym uczestnictwie w rozmowach,
- wysoka centralność pośrednictwa może wskazywać użytkowników łączących różne części dyskusji,
- wykryte społeczności oznaczają grupy użytkowników częściej komunikujących się między sobą niż z resztą sieci.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from pathlib import Path
from collections import Counter
from itertools import combinations

try:
    from community import community_louvain
    LOUVAIN_AVAILABLE = True
except ImportError:
    community_louvain = None
    LOUVAIN_AVAILABLE = False

try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    go = None
    PLOTLY_AVAILABLE = False

FIGURES_DIR = Path('../outputs/figures')
REPORTS_DIR = Path('../outputs/reports')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

DATA_PATH = Path('../data/processed/all_subreddits_sample.csv')
df = pd.read_csv(DATA_PATH)

required_columns = {'author', 'parent_id', 'comment_id', 'subreddit'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Brakuje wymaganych kolumn: {missing_columns}')

# Podstawowe czyszczenie danych sieciowych.
df = df.copy()
df['author'] = df['author'].astype(str)
df['parent_id'] = df['parent_id'].astype(str)
df['comment_id'] = df['comment_id'].astype(str)
df['subreddit'] = df['subreddit'].astype(str)

bot_pattern = r'bot$|automod|moderator|transcrib|helper'
valid_mask = (
    df['author'].notna()
    & ~df['author'].isin(['[deleted]', '[removed]', 'nan', 'None'])
    & ~df['author'].str.lower().str.contains(bot_pattern, regex=True, na=False)
)
df = df.loc[valid_mask].copy()

summary = pd.DataFrame({
    'wartość': [
        len(df),
        df['author'].nunique(),
        df['subreddit'].nunique(),
        ', '.join(sorted(df['subreddit'].unique()))
    ]
}, index=['liczba komentarzy po czyszczeniu', 'liczba unikalnych użytkowników', 'liczba subredditów', 'analizowane subreddity'])

summary.to_csv(REPORTS_DIR / 'network_dataset_summary.csv')
summary

In [ ]:
# Checkpoint configuration
from pathlib import Path
CHECKPOINT_DIR = Path("../data/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR.resolve()}")

In [ ]:
# Save cleaned df checkpoint
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
df_checkpoint_path = CHECKPOINT_DIR / "network_df_checkpoint.pkl"
df.to_pickle(df_checkpoint_path)
print(f"Zapisano df do: {df_checkpoint_path}")
print(f"df shape: {df.shape}")
print(f"Kolumny: {df.columns.tolist()}")

In [ ]:
# Load cleaned df from checkpoint if available
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
df_checkpoint_path = CHECKPOINT_DIR / "network_df_checkpoint.pkl"
if "df" in globals():
    print("df już istnieje w pamięci — pomijam wczytywanie checkpointu.")
elif df_checkpoint_path.exists():
    df = pd.read_pickle(df_checkpoint_path)
    print(f"Wczytano df z: {df_checkpoint_path}")
    print(f"df shape: {df.shape}")
    print(f"Kolumny: {df.columns.tolist()}")
else:
    print(f"Brak checkpointu: {df_checkpoint_path}")

Wstępne czyszczenie usuwa konta techniczne, boty oraz komentarze bez użytecznego autora. Dzięki temu dalsze metryki odnoszą się przede wszystkim do realnych interakcji użytkowników, a nie do automatycznych odpowiedzi systemowych.

## 1. Budowa grafów interakcji

In [ ]:
replies = df[df['parent_id'].str.startswith('t1_', na=False)].copy()
replies['parent_comment_id'] = replies['parent_id'].str.replace('t1_', '', regex=False)

edges = replies.merge(
    df[['comment_id', 'author']].rename(columns={'author': 'parent_author'}),
    left_on='parent_comment_id',
    right_on='comment_id',
    how='inner'
)

# Usuwamy odpowiedzi użytkownika do samego siebie, ponieważ nie tworzą relacji społecznej.
edges = edges[edges['author'] != edges['parent_author']].copy()

# Agregujemy powtarzające się odpowiedzi tej samej pary użytkowników.
weighted_edges = (
    edges
    .groupby(['subreddit', 'author', 'parent_author'], as_index=False)
    .size()
    .rename(columns={'size': 'weight'})
)

weighted_edges.to_csv(REPORTS_DIR / 'weighted_network_edges.csv', index=False)
weighted_edges.head(10)

In [ ]:
graphs = {}
for subreddit, group in weighted_edges.groupby('subreddit'):
    graph = nx.DiGraph()
    for row in group.itertuples(index=False):
        graph.add_edge(row.author, row.parent_author, weight=int(row.weight))
    graphs[subreddit] = graph

network_overview = pd.DataFrame([
    {
        'subreddit': subreddit,
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'weighted_interactions': sum(data['weight'] for _, _, data in graph.edges(data=True))
    }
    for subreddit, graph in graphs.items()
]).sort_values('subreddit')

network_overview.to_csv(REPORTS_DIR / 'network_overview.csv', index=False)
network_overview

In [ ]:
# Load graphs checkpoint if needed
import pickle
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
graphs_checkpoint_path = CHECKPOINT_DIR / "network_graphs_checkpoint.pkl"
if all(name in globals() for name in ["weighted_edges", "graphs", "network_overview"]):
    print("Grafy i krawędzie już istnieją w pamięci — pomijam wczytywanie checkpointu.")
elif graphs_checkpoint_path.exists():
    with open(graphs_checkpoint_path, "rb") as handle:
        checkpoint = pickle.load(handle)
    weighted_edges = checkpoint.get("weighted_edges")
    graphs = checkpoint.get("graphs", {})
    network_overview = checkpoint.get("network_overview")
    print(f"Wczytano grafy z: {graphs_checkpoint_path}")
    print(f"Liczba grafów: {len(graphs)}")
    if hasattr(weighted_edges, "shape"):
        print(f"weighted_edges shape: {weighted_edges.shape}")
    if hasattr(network_overview, "shape"):
        print(f"network_overview shape: {network_overview.shape}")
else:
    print(f"Brak checkpointu: {graphs_checkpoint_path}")

In [ ]:
# Save graphs checkpoint
import pickle
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
graphs_checkpoint_path = CHECKPOINT_DIR / "network_graphs_checkpoint.pkl"
with open(graphs_checkpoint_path, "wb") as handle:
    pickle.dump({
        "weighted_edges": weighted_edges,
        "graphs": graphs,
        "network_overview": network_overview,
    }, handle)
print(f"Zapisano grafy do: {graphs_checkpoint_path}")
print(f"Liczba grafów: {len(graphs)}")
print(f"weighted_edges shape: {weighted_edges.shape}")
print(f"network_overview shape: {network_overview.shape}")

Tabela pokazuje podstawowy rozmiar grafów dla poszczególnych subredditów. Liczba krawędzi oznacza liczbę unikalnych relacji między parami użytkowników, natomiast liczba ważonych interakcji uwzględnia powtarzalność odpowiedzi między tymi samymi osobami.

## 2. Podstawowe statystyki sieci

In [ ]:
def safe_largest_component_size(graph_undirected):
    if graph_undirected.number_of_nodes() == 0:
        return 0
    return len(max(nx.connected_components(graph_undirected), key=len))

basic_metrics = []
for subreddit, graph in graphs.items():
    graph_undirected = graph.to_undirected()
    degrees = [degree for _, degree in graph_undirected.degree()]
    components = list(nx.connected_components(graph_undirected)) if graph_undirected.number_of_nodes() > 0 else []
    basic_metrics.append({
        'subreddit': subreddit,
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'avg_degree': round(float(np.mean(degrees)), 3) if degrees else 0,
        'max_degree': max(degrees) if degrees else 0,
        'density': round(nx.density(graph_undirected), 6),
        'connected_components': len(components),
        'largest_component_size': safe_largest_component_size(graph_undirected),
        'largest_component_share': round(safe_largest_component_size(graph_undirected) / graph.number_of_nodes(), 4) if graph.number_of_nodes() else 0
    })

basic_metrics_df = pd.DataFrame(basic_metrics).sort_values('subreddit')
basic_metrics_df.to_csv(REPORTS_DIR / 'network_basic_metrics.csv', index=False)
basic_metrics_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, (subreddit, graph) in zip(axes, sorted(graphs.items())):
    degrees = [degree for _, degree in graph.to_undirected().degree()]
    if not degrees:
        ax.set_title(f'r/{subreddit} — brak danych')
        ax.axis('off')
        continue
    upper_limit = max(1, int(np.percentile(degrees, 95)))
    ax.hist(degrees, bins=30, range=(0, upper_limit))
    ax.set_title(f'r/{subreddit} — rozkład stopni węzłów')
    ax.set_xlabel('Stopień węzła')
    ax.set_ylabel('Liczba użytkowników')
    ax.set_yscale('log')

for ax in axes[len(graphs):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'degree_distribution_networks.png', dpi=150, bbox_inches='tight')
plt.show()

Rozkład stopni węzłów pozwala ocenić, czy rozmowy są rozproszone między wieloma użytkownikami, czy koncentrują się wokół niewielkiej grupy bardziej aktywnych kont. Skala logarytmiczna ułatwia odczyt, ponieważ w sieciach społecznościowych większość użytkowników ma niewiele połączeń, a tylko nieliczni osiągają bardzo wysoką liczbę relacji.

## 3. Analiza centralności

In [ ]:
def normalize_series(series):
    max_value = series.max()
    if pd.isna(max_value) or max_value == 0:
        return series * 0
    return series / max_value

def calculate_centralities(graph, sample_k=500, seed=42):
    if graph.number_of_nodes() == 0:
        return pd.DataFrame()
    
    degree_centrality = nx.degree_centrality(graph)
    pagerank = nx.pagerank(graph, weight='weight')
    
    k = min(sample_k, max(1, graph.number_of_nodes() - 1))
    betweenness_centrality = nx.betweenness_centrality(graph, k=k, seed=seed, weight=None)
    
    graph_undirected = graph.to_undirected()
    try:
        eigenvector_centrality = nx.eigenvector_centrality(graph_undirected, max_iter=1000, weight='weight')
    except nx.PowerIterationFailedConvergence:
        eigenvector_centrality = {node: np.nan for node in graph.nodes()}
    
    return pd.DataFrame({
        'degree_centrality': pd.Series(degree_centrality),
        'betweenness_centrality': pd.Series(betweenness_centrality),
        'eigenvector_centrality': pd.Series(eigenvector_centrality),
        'pagerank': pd.Series(pagerank)
    })

centrality_tables = {}
for subreddit, graph in graphs.items():
    centrality_df = calculate_centralities(graph)
    centrality_df['subreddit'] = subreddit
    centrality_df.index.name = 'author'
    centrality_tables[subreddit] = centrality_df

all_centralities = pd.concat(centrality_tables.values()).reset_index()
all_centralities.to_csv(REPORTS_DIR / 'network_centralities.csv', index=False)
all_centralities.head()

In [ ]:
for metric in ['degree_centrality', 'betweenness_centrality', 'eigenvector_centrality', 'pagerank']:
    top_metric = (
        all_centralities
        .sort_values(['subreddit', metric], ascending=[True, False])
        .groupby('subreddit')
        .head(10)
        [['subreddit', 'author', metric]]
    )
    top_metric.to_csv(REPORTS_DIR / f'top_users_{metric}.csv', index=False)
    display(top_metric)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, (subreddit, centrality_df) in zip(axes, sorted(centrality_tables.items())):
    top = centrality_df.sort_values('degree_centrality', ascending=False).head(10).iloc[::-1]
    ax.barh(top.index, top['degree_centrality'])
    ax.set_title(f'r/{subreddit} — top 10 degree centrality')
    ax.set_xlabel('Degree centrality')
    ax.set_ylabel('Użytkownik')

for ax in axes[len(centrality_tables):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_degree_centrality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, (subreddit, centrality_df) in zip(axes, sorted(centrality_tables.items())):
    top = centrality_df.sort_values('betweenness_centrality', ascending=False).head(10).iloc[::-1]
    ax.barh(top.index, top['betweenness_centrality'])
    ax.set_title(f'r/{subreddit} — top 10 betweenness centrality')
    ax.set_xlabel('Betweenness centrality')
    ax.set_ylabel('Użytkownik')

for ax in axes[len(centrality_tables):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_betweenness_centrality.png', dpi=150, bbox_inches='tight')
plt.show()

Metryki centralności pokazują różne typy znaczenia użytkowników w sieci. Degree centrality wskazuje osoby mające wiele bezpośrednich połączeń, betweenness centrality identyfikuje użytkowników pośredniczących między częściami sieci, eigenvector centrality premiuje kontakty z innymi ważnymi użytkownikami, a PageRank pomaga wskazać konta, do których prowadzą istotne relacje odpowiedzi.

## 4. Wykrywanie społeczności i klastrów

In [ ]:
community_stats = []
community_assignments = []

for subreddit, graph in graphs.items():
    graph_undirected = graph.to_undirected()
    if graph_undirected.number_of_nodes() == 0:
        continue
    
    if LOUVAIN_AVAILABLE:
        partition = community_louvain.best_partition(graph_undirected, weight='weight', random_state=42)
        modularity = community_louvain.modularity(partition, graph_undirected, weight='weight')
        method = 'Louvain'
    else:
        communities = nx.algorithms.community.greedy_modularity_communities(graph_undirected, weight='weight')
        partition = {node: idx for idx, community in enumerate(communities) for node in community}
        modularity = nx.algorithms.community.modularity(graph_undirected, communities, weight='weight')
        method = 'greedy_modularity'
    
    nx.set_node_attributes(graph, partition, 'community')
    counts = Counter(partition.values())
    largest_communities = counts.most_common(5)
    
    community_stats.append({
        'subreddit': subreddit,
        'method': method,
        'communities': len(counts),
        'modularity': round(float(modularity), 4),
        'largest_community_size': largest_communities[0][1] if largest_communities else 0,
        'largest_community_share': round(largest_communities[0][1] / graph.number_of_nodes(), 4) if largest_communities else 0
    })
    
    for author, community_id in partition.items():
        community_assignments.append({
            'subreddit': subreddit,
            'author': author,
            'community': community_id
        })

community_stats_df = pd.DataFrame(community_stats).sort_values('subreddit')
community_assignments_df = pd.DataFrame(community_assignments)
community_stats_df.to_csv(REPORTS_DIR / 'community_detection_stats.csv', index=False)
community_assignments_df.to_csv(REPORTS_DIR / 'community_assignments.csv', index=False)
community_stats_df

In [ ]:
# Load community detection checkpoint if needed
import pickle
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
community_checkpoint_path = CHECKPOINT_DIR / "community_detection_checkpoint.pkl"
if all(name in globals() for name in ["graphs", "community_stats_df", "community_assignments_df"]):
    print("Dane społeczności już istnieją w pamięci — pomijam wczytywanie checkpointu.")
elif community_checkpoint_path.exists():
    with open(community_checkpoint_path, "rb") as handle:
        checkpoint = pickle.load(handle)
    graphs = checkpoint.get("graphs", {})
    community_stats_df = checkpoint.get("community_stats_df")
    community_assignments_df = checkpoint.get("community_assignments_df")
    print(f"Wczytano wyniki społeczności z: {community_checkpoint_path}")
    print(f"Liczba grafów: {len(graphs)}")
    if hasattr(community_stats_df, "shape"):
        print(f"community_stats_df shape: {community_stats_df.shape}")
    if hasattr(community_assignments_df, "shape"):
        print(f"community_assignments_df shape: {community_assignments_df.shape}")
else:
    print(f"Brak checkpointu: {community_checkpoint_path}")

In [ ]:
# Save community detection checkpoint
import pickle
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
community_checkpoint_path = CHECKPOINT_DIR / "community_detection_checkpoint.pkl"
with open(community_checkpoint_path, "wb") as handle:
    pickle.dump({
        "graphs": graphs,
        "community_stats_df": community_stats_df,
        "community_assignments_df": community_assignments_df,
    }, handle)
print(f"Zapisano wyniki społeczności do: {community_checkpoint_path}")
print(f"Liczba grafów: {len(graphs)}")
print(f"community_stats_df shape: {community_stats_df.shape}")
print(f"community_assignments_df shape: {community_assignments_df.shape}")

In [ ]:
largest_communities_summary = []
for subreddit, group in community_assignments_df.groupby('subreddit'):
    counts = group['community'].value_counts().head(10)
    for community_id, size in counts.items():
        largest_communities_summary.append({
            'subreddit': subreddit,
            'community': community_id,
            'size': size
        })

largest_communities_df = pd.DataFrame(largest_communities_summary)
largest_communities_df.to_csv(REPORTS_DIR / 'largest_communities.csv', index=False)
largest_communities_df

Wykrywanie społeczności pokazuje, czy dyskusje tworzą zwarte grupy użytkowników. Wysoka modularność sugeruje silniejszy podział na klastry, natomiast niższa modularność oznacza bardziej wymieszaną strukturę rozmów. Wielkość największej społeczności pozwala ocenić, czy aktywność skupia się w jednym głównym jądrze, czy jest rozproszona między kilka grup.

## 5. Wizualizacja grafów

In [ ]:
def get_visualization_subgraph(graph, max_nodes=120):
    graph_undirected = graph.to_undirected()
    if graph_undirected.number_of_nodes() == 0:
        return graph_undirected
    largest_component = max(nx.connected_components(graph_undirected), key=len)
    subgraph = graph_undirected.subgraph(largest_component).copy()
    if subgraph.number_of_nodes() > max_nodes:
        top_nodes = [node for node, _ in sorted(subgraph.degree(weight='weight'), key=lambda x: x[1], reverse=True)[:max_nodes]]
        subgraph = subgraph.subgraph(top_nodes).copy()
    return subgraph

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (subreddit, graph) in zip(axes, sorted(graphs.items())):
    subgraph = get_visualization_subgraph(graph, max_nodes=120)
    if subgraph.number_of_nodes() == 0:
        ax.set_title(f'r/{subreddit} — brak danych')
        ax.axis('off')
        continue
    
    positions = nx.spring_layout(subgraph, seed=42, k=0.7)
    communities = nx.get_node_attributes(graph, 'community')
    node_colors = [communities.get(node, 0) for node in subgraph.nodes()]
    node_degrees = dict(subgraph.degree(weight='weight'))
    node_sizes = [30 + node_degrees[node] * 8 for node in subgraph.nodes()]
    edge_weights = [subgraph[u][v].get('weight', 1) for u, v in subgraph.edges()]
    edge_widths = [0.2 + min(weight, 10) * 0.15 for weight in edge_weights]
    
    nx.draw_networkx_edges(subgraph, positions, ax=ax, alpha=0.25, width=edge_widths)
    nx.draw_networkx_nodes(
        subgraph,
        positions,
        ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        cmap=plt.cm.tab20,
        alpha=0.85
    )
    
    # Etykiety pokazujemy tylko dla kilku najbardziej połączonych użytkowników.
    top_labels = dict(sorted(node_degrees.items(), key=lambda x: x[1], reverse=True)[:8])
    label_positions = {node: positions[node] for node in top_labels}
    nx.draw_networkx_labels(subgraph, label_positions, labels={node: node for node in top_labels}, font_size=8, ax=ax)
    
    ax.set_title(f'r/{subreddit} — największa składowa / top użytkownicy')
    ax.axis('off')

for ax in axes[len(graphs):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'network_communities_readable.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if PLOTLY_AVAILABLE:
    for subreddit, graph in sorted(graphs.items()):
        subgraph = get_visualization_subgraph(graph, max_nodes=60)
        if subgraph.number_of_nodes() < 2:
            continue
        positions = nx.spring_layout(subgraph, dim=3, seed=42, k=0.8)
        communities = nx.get_node_attributes(graph, 'community')
        degrees = dict(subgraph.degree(weight='weight'))
        
        edge_x, edge_y, edge_z = [], [], []
        for source, target in subgraph.edges():
            x0, y0, z0 = positions[source]
            x1, y1, z1 = positions[target]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]
        
        edge_trace = go.Scatter3d(
            x=edge_x,
            y=edge_y,
            z=edge_z,
            mode='lines',
            line=dict(width=1),
            hoverinfo='none'
        )
        
        node_trace = go.Scatter3d(
            x=[positions[node][0] for node in subgraph.nodes()],
            y=[positions[node][1] for node in subgraph.nodes()],
            z=[positions[node][2] for node in subgraph.nodes()],
            mode='markers+text',
            text=[node if degree >= sorted(degrees.values(), reverse=True)[min(9, len(degrees)-1)] else '' for node, degree in degrees.items()],
            hovertext=[f'{node}<br>degree: {degrees[node]}<br>community: {communities.get(node, "brak")}' for node in subgraph.nodes()],
            hoverinfo='text',
            marker=dict(
                size=[6 + min(degrees[node], 20) for node in subgraph.nodes()],
                color=[communities.get(node, 0) for node in subgraph.nodes()],
                opacity=0.85
            )
        )
        
        figure = go.Figure(data=[edge_trace, node_trace])
        figure.update_layout(
            title=f'r/{subreddit} — interaktywna wizualizacja sieci',
            showlegend=False,
            margin=dict(l=0, r=0, b=0, t=40)
        )
        figure.write_html(FIGURES_DIR / f'network_3d_{subreddit}.html')
        figure.show()
else:
    print('Plotly nie jest zainstalowane, dlatego pominięto interaktywną wizualizację 3D.')

Wizualizacje zostały ograniczone do największych i najbardziej aktywnych fragmentów grafów, ponieważ pełne sieci użytkowników są zwykle zbyt gęste i nieczytelne. Wielkość węzłów odzwierciedla ważony stopień użytkownika, a kolory oznaczają wykryte społeczności.

## 6. Cross-subreddit: wspólni użytkownicy

In [ ]:
subreddit_users = df.groupby('subreddit')['author'].apply(set)
subreddits = sorted(subreddit_users.index)

overlap_rows = []
for subreddit_a, subreddit_b in combinations(subreddits, 2):
    overlap = subreddit_users[subreddit_a] & subreddit_users[subreddit_b]
    overlap_rows.append({
        'subreddit_a': subreddit_a,
        'subreddit_b': subreddit_b,
        'shared_users': len(overlap),
        'share_of_a': round(len(overlap) / len(subreddit_users[subreddit_a]), 4) if len(subreddit_users[subreddit_a]) else 0,
        'share_of_b': round(len(overlap) / len(subreddit_users[subreddit_b]), 4) if len(subreddit_users[subreddit_b]) else 0
    })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(REPORTS_DIR / 'cross_subreddit_user_overlap.csv', index=False)
overlap_df

In [ ]:
matrix = np.zeros((len(subreddits), len(subreddits)))
for i, subreddit_a in enumerate(subreddits):
    for j, subreddit_b in enumerate(subreddits):
        if i == j:
            matrix[i, j] = len(subreddit_users[subreddit_a])
        else:
            matrix[i, j] = len(subreddit_users[subreddit_a] & subreddit_users[subreddit_b])

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(matrix)
ax.set_xticks(range(len(subreddits)))
ax.set_yticks(range(len(subreddits)))
ax.set_xticklabels([f'r/{subreddit}' for subreddit in subreddits], rotation=45, ha='right')
ax.set_yticklabels([f'r/{subreddit}' for subreddit in subreddits])

for i in range(len(subreddits)):
    for j in range(len(subreddits)):
        ax.text(j, i, f'{int(matrix[i, j]):,}', ha='center', va='center', fontsize=9)

ax.set_title('Liczba wspólnych użytkowników między subredditami')
plt.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cross_subreddit_user_overlap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
user_subreddit_count = df.groupby('author')['subreddit'].nunique()
user_subreddit_distribution = user_subreddit_count.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(user_subreddit_distribution.index, user_subreddit_distribution.values)
ax.set_title('Liczba subredditów, w których aktywny był użytkownik')
ax.set_xlabel('Liczba subredditów')
ax.set_ylabel('Liczba użytkowników')
ax.set_xticks(user_subreddit_distribution.index)

for x, y in zip(user_subreddit_distribution.index, user_subreddit_distribution.values):
    ax.text(x, y, f'{y:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'user_subreddit_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

users_in_all_subreddits = set.intersection(*subreddit_users.tolist()) if len(subreddit_users) > 0 else set()
users_in_all_df = (
    df[df['author'].isin(users_in_all_subreddits)]
    .groupby('author')
    .agg(total_comments=('comment_id', 'count'), subreddits=('subreddit', lambda x: ', '.join(sorted(set(x)))))
    .sort_values('total_comments', ascending=False)
)
users_in_all_df.to_csv(REPORTS_DIR / 'users_active_in_all_subreddits.csv')
users_in_all_df.head(20)

Analiza wspólnych użytkowników pozwala sprawdzić, czy społeczności muzyczne są od siebie odseparowane, czy też występuje między nimi przepływ uczestników. Użytkownicy aktywni w kilku subredditach mogą pełnić rolę pomostów między różnymi fandomami muzycznymi, choć sama obecność w wielu grupach nie oznacza jeszcze wysokiej pozycji w sieci interakcji.

## 7. Użytkownicy o najwyższej centralności w sieci

W tej części zamiast określenia „influencerzy” stosowane jest bardziej neutralne pojęcie „użytkownicy o najwyższej centralności”. Wskaźnik `influence_score` ma charakter pomocniczy i autorski. Jest liczony jako średnia z trzech znormalizowanych miar: degree centrality, betweenness centrality oraz liczby komentarzy autora w danym subreddicie.

Ograniczenie tego wskaźnika polega na tym, że nie mierzy on realnego wpływu społecznego, jakości wypowiedzi ani odbioru komentarzy przez innych użytkowników. Pokazuje jedynie, którzy użytkownicy jednocześnie często komentują, mają wiele relacji i mogą łączyć różne części sieci.

In [ ]:
comment_counts = (
    df.groupby(['subreddit', 'author'])['comment_id']
    .count()
    .rename('comment_count')
    .reset_index()
)

influence_tables = []
for subreddit, centrality_df in centrality_tables.items():
    temp = centrality_df.reset_index().merge(
        comment_counts[comment_counts['subreddit'] == subreddit],
        on=['subreddit', 'author'],
        how='left'
    )
    temp['comment_count'] = temp['comment_count'].fillna(0)
    
    for column in ['degree_centrality', 'betweenness_centrality', 'comment_count']:
        temp[f'{column}_norm'] = normalize_series(temp[column])
    
    temp['influence_score'] = temp[[
        'degree_centrality_norm',
        'betweenness_centrality_norm',
        'comment_count_norm'
    ]].mean(axis=1)
    influence_tables.append(temp)

influence_df = pd.concat(influence_tables, ignore_index=True)
influence_df.to_csv(REPORTS_DIR / 'network_influence_score.csv', index=False)

top_influence_df = (
    influence_df
    .sort_values(['subreddit', 'influence_score'], ascending=[True, False])
    .groupby('subreddit')
    .head(10)
)
top_influence_df.to_csv(REPORTS_DIR / 'top_structurally_central_users.csv', index=False)
top_influence_df[['subreddit', 'author', 'degree_centrality', 'betweenness_centrality', 'comment_count', 'influence_score']]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, sorted(top_influence_df.groupby('subreddit'))):
    top = group.sort_values('influence_score', ascending=True)
    ax.barh(top['author'], top['influence_score'])
    ax.set_title(f'r/{subreddit} — użytkownicy o najwyższej centralności')
    ax.set_xlabel('Pomocniczy influence score')
    ax.set_ylabel('Użytkownik')

for ax in axes[top_influence_df['subreddit'].nunique():]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_structurally_central_users.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, (subreddit, group) in zip(axes, sorted(influence_df.groupby('subreddit'))):
    top_group = group.sort_values('influence_score', ascending=False).head(25)
    sizes = top_group['betweenness_centrality_norm'].fillna(0) * 500 + 30
    ax.scatter(top_group['degree_centrality'], top_group['comment_count'], s=sizes, alpha=0.7)
    
    for _, row in top_group.head(5).iterrows():
        ax.annotate(row['author'], (row['degree_centrality'], row['comment_count']), textcoords='offset points', xytext=(5, 5), fontsize=8)
    
    ax.set_title(f'r/{subreddit} — centralność a aktywność')
    ax.set_xlabel('Degree centrality')
    ax.set_ylabel('Liczba komentarzy')

for ax in axes[influence_df['subreddit'].nunique():]:
    ax.axis('off')

plt.suptitle('Rozmiar punktu odpowiada betweenness centrality', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'centrality_vs_activity.png', dpi=150, bbox_inches='tight')
plt.show()

Najwyższy wynik pomocniczego wskaźnika centralności uzyskują użytkownicy łączący aktywność komentarzową z silną pozycją w grafie odpowiedzi. W interpretacji należy jednak zachować ostrożność: wysoka centralność nie musi oznaczać autorytetu, popularności ani pozytywnego odbioru przez społeczność.

## Wnioski z analizy sieci społecznościowych

Analiza sieciowa umożliwia spojrzenie na społeczności muzyczne Reddita nie tylko przez treść komentarzy, lecz także przez strukturę interakcji między użytkownikami. Graf odpowiedzi pokazuje, które subreddity mają bardziej rozproszony model rozmów, a które skupiają się wokół mniejszej liczby aktywnych uczestników.

Podstawowe statystyki sieci, takie jak liczba węzłów, liczba krawędzi, gęstość i wielkość największej składowej, pozwalają porównać intensywność komunikacji w poszczególnych społecznościach. Metryki centralności wskazują użytkowników zajmujących ważne pozycje strukturalne, natomiast analiza społeczności pokazuje, czy w obrębie subredditów powstają wyraźne klastry dyskusyjne.

Wyniki należy interpretować w kontekście specyfiki Reddita. Relacje w grafie wynikają wyłącznie z odpowiedzi w komentarzach, dlatego nie obejmują biernego czytania, głosowania, reputacji użytkowników ani jakości wypowiedzi. Mimo tych ograniczeń analiza sieci dobrze uzupełnia wcześniejszą eksplorację danych i text mining, ponieważ pokazuje społeczną strukturę dyskusji wokół różnych gatunków muzycznych.